In [ ]:
# Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE


# Load dataset
file_path = "MACHINE HEALTH & MAINTENANCE.xlsx"
df = pd.read_excel(file_path)

# Remove missing values
df = df.dropna()

# Target column
target_column = "MaintenanceFlag"

X = df.drop(columns=[target_column])
y = df[target_column]


# DATETIME FEATURE EXTRACTION 
datetime_cols = X.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns

for col in datetime_cols:
    X[col] = pd.to_datetime(X[col])
    X[col + "_year"] = X[col].dt.year
    X[col + "_month"] = X[col].dt.month
    X[col + "_day"] = X[col].dt.day
    X[col + "_hour"] = X[col].dt.hour
    X = X.drop(columns=[col])


# ENCODE CATEGORICAL VARIABLES 
categorical_cols = X.select_dtypes(include=['object']).columns

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le


# Encode target if needed
if y.dtype == 'object':
    y = LabelEncoder().fit_transform(y)


#  TRAIN TEST SPLIT 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


#  HANDLE CLASS IMBALANCE USING SMOTE 
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)


#  RANDOM FOREST MODEL 
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train_res, y_train_res)


#  MODEL PREDICTIONS 
rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]


#  MODEL PERFORMANCE 
print("\n----- RANDOM FOREST RESULTS -----")
print("Accuracy:", accuracy_score(y_test, rf_preds))
print("Precision:", precision_score(y_test, rf_preds))
print("Recall:", recall_score(y_test, rf_preds))
print("F1 Score:", f1_score(y_test, rf_preds))


#  FIND TOP 20 HIGH RISK MACHINES 
future_data = X_test.copy()
future_data["Predicted_Probability"] = rf_probs

top_20_risk = future_data.sort_values(
    by="Predicted_Probability",
    ascending=False
).head(20)

print("\n----- TOP 20 HIGH-RISK ROWS (Random Forest) -----")
print(top_20_risk)


# Save result
top_20_risk.to_excel("Top_20_High_Risk_Machines_RF.xlsx", index=False)

c:\Users\sahan\anaconda3\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(



----- RANDOM FOREST RESULTS -----
Accuracy: 0.7685
Precision: 0.10244648318042814
Recall: 0.16502463054187191
F1 Score: 0.12641509433962264

----- TOP 20 HIGH-RISK ROWS (Random Forest) -----
      MachineID  Plant  Temperature  Vibration   Pressure  EnergyConsumption  \
2700        110      1    65.865311   4.711285  30.008221         314.587752   
2173        138      0    65.639462   4.574214  33.730895         217.311791   
4215        136      1    72.627121   4.598192  32.405617         233.050747   
3391        116      0    63.258375   5.517819  27.645591         287.673062   
3704        125      1    63.976414   4.376608  32.200144         206.772708   
2722        123      0    66.290578   5.632460  32.482914         250.706959   
4068        125      1    61.846960   5.489577  27.210568         229.774266   
2646        146      0    77.911251   5.068782  34.655648         281.901946   
1712        146      1    70.238112   4.370061  29.816775         226.879398   
1758    